# Drift quickstart

Build containers, wire them into a simulation, and integrate.

## Potentials

Create a potential with the static constructors on `dft.Potential`.

In [ ]:
import drift as dft
import numpy as np

kepler = dft.Potential.kepler(amp=1.0)  # point-mass potential (G = 1); amp is the total mass

## Containers

Containers are a background feature plus groups of particles. The initial
state is an `(N, 6)` array (or flat `6N`) with phase-space columns
`[x, y, z, vx, vy, vz]`.

In [ ]:
bg = dft.background(kepler)
gmc = dft.particles(kepler, np.array([[1.0, 0.0, 0.0, 0.0, 1.0, 0.0]]))
iso = dft.test_particles(np.array([[-1.0, 0.0, 0.0, 0.0, -1.0, 0.0]]))

## Simulation

A `Config` bundles the backend, scheme, and output times; `add` wires each
container together with the containers it depends on.

In [ ]:
test_sim = dft.Config(
    engine=dft.Engine.CPU,
    method=dft.Method.DOPR54,
    variant=dft.Variant.Compatible,
    ts=(0.0, 100.0, 201),
)
test_sim.add(gmc, bg)        # gmc is integrated with bg as an input
test_sim.add(iso, gmc, bg)

## Integrate

`run` returns one `(N, 11)` float64 array per particle group, aligned with
the integration order; background containers contribute `None`.

In [ ]:
results = test_sim.run()
for i, frame in enumerate(results):
    print(f"group {i}: {'None (background)' if frame is None else frame.shape}")

## Next steps

See the docstrings (`help(dft.Config)`) and
[Testing_Instructions.md](../docs/Testing_Instructions.md) for details.